# Speculatores 15 Colab Runner

This notebook is a thin front end for the script-backed `Speculatores 15` pipeline.
Run top to bottom. The optimizer runs in the background, writes a log, and the monitor cell shows live progress for both sides.

In [ ]:
# Cell 1 ? Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 2 ? Clone or update repo and install deps
import os
from pathlib import Path
from datetime import datetime

REPO_DIR = Path('/content/cfd9')
REPO_URL = 'https://github.com/Sovenski/cfd9.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git rev-parse HEAD


In [ ]:
# Cell 3 - Run config
from pathlib import Path
from datetime import datetime
import re

DRIVE_ROOT = Path('/content/drive/MyDrive/cfd9')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_OPTIONS = {
    'SPX_1D': DRIVE_ROOT / 'data/raw/SPX_1D_18710201_20260318.csv',
    'DAX_1M': DRIVE_ROOT / 'data/raw/DAX_1M_20250113_20260227.csv',
}
DATASET_KEY = 'SPX_1D'
DATASET = str(DATASET_OPTIONS[DATASET_KEY])

TRIALS_PER_SIDE = 250
WORKERS_PER_SIDE = 4
STARTUP_TRIALS = 80
STABILITY_TRIALS = 50
SEED = 42
SKIP_CROSS_ASSET = False
RESUME_EXISTING = False

run_label = re.sub(r'[^a-z0-9]+', '_', DATASET_KEY.lower()).strip('_')
RUN_SLUG = f"{run_label}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
STUDY_PREFIX = f'speculatores_15_{RUN_SLUG}'

ACTIVE_ROOT = Path('/content/spec145_runs')
ACTIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = ACTIVE_ROOT / RUN_SLUG
RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_DIR = DRIVE_ROOT / 'runs' / RUN_SLUG
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
STORAGE = RUN_DIR / 'spec145.journal'
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / 'spec145.log'
PID_PATH = RUN_DIR / 'spec145.pid'

assert Path(DATASET).exists(), f'Missing dataset: {DATASET}'

print({
    'dataset_key': DATASET_KEY,
    'dataset': DATASET,
    'dataset_options': {k: str(v) for k, v in DATASET_OPTIONS.items()},
    'trials_per_side': TRIALS_PER_SIDE,
    'workers_per_side': WORKERS_PER_SIDE,
    'startup_trials': STARTUP_TRIALS,
    'stability_trials': STABILITY_TRIALS,
    'resume_existing': RESUME_EXISTING,
    'run_slug': RUN_SLUG,
    'run_dir': str(RUN_DIR),
    'drive_run_dir': str(DRIVE_RUN_DIR),
    'storage': str(STORAGE),
    'results_dir': str(RESULTS_DIR),
    'log_path': str(LOG_PATH),
    'pid_path': str(PID_PATH),
    'skip_cross_asset': SKIP_CROSS_ASSET,
})



In [ ]:
# Cell 4 ? Launch Speculatores 15 in background
import os
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/cfd9').resolve()
assert REPO_DIR.exists(), REPO_DIR
assert (REPO_DIR / 'scripts' / 'run_speculatores_145.py').exists()

cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'run_speculatores_145.py'),
    '--dataset', str(DATASET),
    '--trials-per-side', str(TRIALS_PER_SIDE),
    '--workers-per-side', str(WORKERS_PER_SIDE),
    '--startup-trials', str(STARTUP_TRIALS),
    '--stability-trials', str(STABILITY_TRIALS),
    '--seed', str(SEED),
    '--study-prefix', str(STUDY_PREFIX),
    '--storage', str(STORAGE),
    '--results-dir', str(RESULTS_DIR),
]
if SKIP_CROSS_ASSET:
    cmd.append('--skip-cross-asset')

if not RESUME_EXISTING and STORAGE.exists():
    STORAGE.unlink()

if PID_PATH.exists():
    try:
        old_pid = int(PID_PATH.read_text().strip())
        os.kill(old_pid, 0)
        raise RuntimeError(f'Run already active with PID {old_pid}. Stop it first or delete {PID_PATH}.')
    except OSError:
        PID_PATH.unlink(missing_ok=True)

log_handle = open(LOG_PATH, 'w', encoding='utf-8')
proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)
PID_PATH.write_text(str(proc.pid), encoding='utf-8')
print('RUNNING:')
print(' '.join(cmd))
print(f'PID: {proc.pid}')
print(f'Log: {LOG_PATH}')


In [ ]:
# Cell 5 ? Monitor optimizer progress
import time
from pathlib import Path
from IPython.display import clear_output
from src.monitor145 import summarize_run, progress_dict


def _dataset_version_from_path(path: str) -> str:
    return Path(path).stem


def _bar(done: int, total: int, width: int = 24) -> str:
    total = max(total, 1)
    done = min(done, total)
    filled = int(width * done / total)
    return '[' + '#' * filled + '-' * (width - filled) + f'] {done}/{total}'


dataset_version = _dataset_version_from_path(DATASET)

for _ in range(10_000):
    clear_output(wait=True)
    pid_text = PID_PATH.read_text().strip() if PID_PATH.exists() else 'not running'
    print(f'PID: {pid_text}')
    print(f'Storage: {STORAGE}')
    print(f'Log: {LOG_PATH}')
    print('')
    try:
        summary = summarize_run(STORAGE, STUDY_PREFIX, dataset_version, TRIALS_PER_SIDE)
        total = progress_dict(summary)
        print('OPTIMIZER')
        print(_bar(total['total_done'], total['target']))
        print(f"running={total['running']} complete={total['completed']} pruned={total['pruned']} failed={total['failed']}")
        print('')
        for side, study in (('HIGH', summary.high), ('LOW', summary.low)):
            info = progress_dict(study)
            print(f"{side}: best={info['best_value']} p75={info['p75_value']}")
    except Exception as exc:
        print(f'Progress unavailable yet: {exc}')
        print('')

    if LOG_PATH.exists():
        print('LOG TAIL')
        lines = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-30:]))
        if any('Speculatores 15 report written to:' in line for line in lines[-30:]):
            PID_PATH.unlink(missing_ok=True)
            print('\nRun process finished.')
            break

    if PID_PATH.exists():
        try:
            import os
            os.kill(int(PID_PATH.read_text().strip()), 0)
        except OSError:
            PID_PATH.unlink(missing_ok=True)
            print('\nRun process finished.')
            break
    else:
        print('\nRun process finished.')
        break
    time.sleep(10)


In [ ]:
# Cell 6 ? Show this run's report and preview
from pathlib import Path

reports = sorted(Path(RESULTS_DIR).glob(f'*{Path(DATASET).stem}*__speculatores_15_pathA.md'), key=lambda p: p.stat().st_mtime, reverse=True)
assert reports, 'No reports found.'

current_run_reports = [p for p in reports if RUN_SLUG in p.name]
report_path = current_run_reports[0] if current_run_reports else reports[0]
print(f'Report: {report_path}')
print(report_path.read_text(encoding='utf-8'))


In [ ]:
# Cell 7 ? Optional: inspect parity section only
text = report_path.read_text(encoding='utf-8')
marker = '## Cell 3.3'
idx = text.find(marker)
if idx >= 0:
    print(text[idx:idx+2500])
else:
    print('Parity section not found.')
